# **Transformers**

In [ ]:
import torch
import torch.nn as nn

# 1. REAL (YET TINY) TEXT DATASET
data = [
    ("i love this movie", 1), ("this is an amazing film", 1), ("absolutely fantastic", 1),
    ("i hate this movie", 0), ("this is a terrible film", 0), ("absolutely awful", 0)
]

# Simple tokenization & Vocabulary mapping
vocab = {"<pad>": 0}
for text, _ in data:
    for word in text.split():
        if word not in vocab: vocab[word] = len(vocab)

# Convert text to padded tensors of uniform length (size=4)
X = torch.tensor([[vocab[w] for w in txt.split()] + [0]*(4-len(txt.split())) for txt, _ in data])
Y = torch.tensor([label for _, label in data], dtype=torch.float32)

# 2. MINIMAL TRANSFORMER CLASSIFIER
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=16, nhead=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        # Using a single built-in Transformer Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc_out = nn.Linear(d_model, 1) # Binary classification output

    def forward(self, x):
        x = self.embedding(x)             # [Batch, Seq_Len, d_model]
        x = self.transformer(x)           # Process tokens simultaneously
        x = x.mean(dim=1)                 # Pool sequence tokens into one vector
        return torch.sigmoid(self.fc_out(x)).squeeze()

# 3. SHORT TRAINING LOOP
model = TinyTransformer(vocab_size=len(vocab))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(40):
    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, Y)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/40, Loss: {loss.item():.4f}")

# 4. INFERENCE (TESTING)
model.eval()
with torch.no_grad():
    # Helper to clean/tokenize novel data based on our built mini-vocab
    test_phrase = "amazing movie"
    test_input = torch.tensor([[vocab.get(w, 0) for w in test_phrase.split()]])
    
    prob = model(test_input).item()
    print(f"\nReview: '{test_phrase}' -> Sentiment Score: {prob:.4f} ({'Positive' if prob > 0.5 else 'Negative'})")